In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("../data/terremoti_ingv_1985_2025.csv")

In [3]:
data

,time,magnitude,place,depth,latitude,longitude
0,1985-01-02T22:57:43.090000,2.6,6 km S Civitella Alfedena (AQ),5.8,41.7150,13.9410
1,1985-01-03T00:46:16.990000,2.5,6 km NE Picinisco (FR),6.1,41.6770,13.9310
2,1985-01-05T08:32:41.900000,2.6,3 km NW Fiastra (MC),2.8,43.0560,13.1390
3,1985-01-05T10:13:26.800000,2.2,1 km E San Raffaele Cimena (TO),20.8,45.1500,7.8600
4,1985-01-06T03:52:23.380000,2.4,3 km W Gubbio (PG),10.0,43.3570,12.5370
...,...,...,...,...,...,...
441817,2024-12-31T22:00:34.100000,1.2,5 km NE Barberino di Mugello (FI),10.2,44.0287,11.2822
441818,2024-12-31T22:51:12.920000,1.1,4 km NE Barberino di Mugello (FI),10.3,44.0262,11.2815
441819,2024-12-31T23:29:28.530000,1.0,5 km NE Barberino di Mugello (FI),9.1,44.0267,11.2852
441820,2024-12-31T23:39:56.600000,0.7,4 km NE Barberino di Mugello (FI),10.1,44.0273,11.2777


### Aggiunta di features temporali

In [ ]:
# Converte la colonna "time" in oggetti datetime (utile per l'estrazione di informazioni temporali)
data["time"] = pd.to_datetime(data["time"])

# Estrae l'anno dalla colonna "time"
data['year'] = data['time'].dt.year

# Estrae il mese dalla colonna "time"
data['month'] = data['time'].dt.month

# Estrae il giorno del mese dalla colonna "time"
data['day'] = data['time'].dt.day

# Estrae l'ora dalla colonna "time"
data['hour'] = data['time'].dt.hour

# Crea una nuova colonna 'period' che indica se l'orario è di giorno (8-20) o di notte
data['period'] = data['hour'].apply(lambda x: 'day' if 8 <= x < 20 else 'night')

In [ ]:

def get_season(date):
    """
    Restituisce la stagione dell'anno in base alla data fornita.

    Args:
        date (datetime): una data in formato datetime.

    Returns:
        str: la stagione corrispondente ('spring', 'summer', 'autumn', 'winter')
             secondo le convenzioni astronomiche per l'emisfero nord.
    """
    if (date.month == 3 and date.day >= 21) or (3 < date.month < 6) or (date.month == 6 and date.day <= 20):
        return 'spring'
    elif (date.month == 6 and date.day >= 21) or (6 < date.month < 9) or (date.month == 9 and date.day <= 21):
        return 'summer'
    elif (date.month == 9 and date.day >= 22) or (9 < date.month < 12) or (date.month == 12 and date.day <= 20):
        return 'autumn'
    else:
        return 'winter'

# Applica la funzione get_season alla colonna 'time' per ottenere la stagione di ogni riga
data['season'] = data['time'].apply(get_season)


In [ ]:
# Definisce gli intervalli di anni per il raggruppamento in decenni (intervalli chiusi a destra)
bins = [1984, 1994, 2004, 2014, 2024]

# Etichette da assegnare a ciascun intervallo di decenni
labels = ['1985–1994', '1995–2004', '2005–2014', '2015–2024']

# Crea una nuova colonna 'decade' assegnando a ogni anno la rispettiva etichetta di decennio
# Utilizza pd.cut per suddividere gli anni in intervalli specificati da 'bins'
data['decade'] = pd.cut(data['year'], bins=bins, labels=labels, right=True)

In [7]:
data

,time,magnitude,place,depth,latitude,longitude,year,month,day,hour,period,season,decade
0,1985-01-02 22:57:43.090,2.6,6 km S Civitella Alfedena (AQ),5.8,41.7150,13.9410,1985,1,2,22,night,winter,1985–1994
1,1985-01-03 00:46:16.990,2.5,6 km NE Picinisco (FR),6.1,41.6770,13.9310,1985,1,3,0,night,winter,1985–1994
2,1985-01-05 08:32:41.900,2.6,3 km NW Fiastra (MC),2.8,43.0560,13.1390,1985,1,5,8,day,winter,1985–1994
3,1985-01-05 10:13:26.800,2.2,1 km E San Raffaele Cimena (TO),20.8,45.1500,7.8600,1985,1,5,10,day,winter,1985–1994
4,1985-01-06 03:52:23.380,2.4,3 km W Gubbio (PG),10.0,43.3570,12.5370,1985,1,6,3,night,winter,1985–1994
...,...,...,...,...,...,...,...,...,...,...,...,...,...
441817,2024-12-31 22:00:34.100,1.2,5 km NE Barberino di Mugello (FI),10.2,44.0287,11.2822,2024,12,31,22,night,winter,2015–2024
441818,2024-12-31 22:51:12.920,1.1,4 km NE Barberino di Mugello (FI),10.3,44.0262,11.2815,2024,12,31,22,night,winter,2015–2024
441819,2024-12-31 23:29:28.530,1.0,5 km NE Barberino di Mugello (FI),9.1,44.0267,11.2852,2024,12,31,23,night,winter,2015–2024
441820,2024-12-31 23:39:56.600,0.7,4 km NE Barberino di Mugello (FI),10.1,44.0273,11.2777,2024,12,31,23,night,winter,2015–2024


### Aggiunta di features spaziali

In [ ]:
# Estrae il contenuto tra parentesi tonde nella colonna 'place' (es. "Napoli (NA)" → "NA")
# e lo assegna alla nuova colonna 'province'
data['province'] = data['place'].str.extract(r'\((.*?)\)')[0]

In [9]:
data[data['province'].isna()]['place'].unique()

array(['Albania', 'Bosnia and Herz. [Land]', 'Albania [Land]',
       'Macedonia [Land]', 'Croatia [Land]', 'Montenegro [Land]',
       'Hungary', 'Kosovo [Land]', 'France',
       'Near south coast of France [Land: France]', 'Serbia [Land]',
       'Albania [Sea]', 'Hungary [Land]', 'San Marino [Land]', 'Vesuvio',
       'Campi Flegrei', 'Ischia'], dtype=object)

In [ ]:
# Per i luoghi noti che non riportano la provincia tra parentesi,
# assegna manualmente la provincia 'NA' (Napoli) se attualmente è mancante
data.loc[
    (data['place'].isin(['Campi Flegrei', 'Vesuvio', 'Ischia'])) & (data['province'].isna()),
    'province'
] = 'NA'

In [11]:
data['province'].unique()

array(['AQ', 'FR', 'MC', 'TO', 'PG', 'KR', 'FRANCIA', 'Catania', 'UD',
       'Foggia', 'VE', 'RI', 'CS', 'EN', nan, 'FI', 'PZ', 'BG', 'VB',
       'SA', 'FG', 'ALBANIA', 'LE', 'TR', 'Cosenza', 'MT',
       'SLOVENIA, CROAZIA', 'MS', 'BN', 'MARE', 'PU', 'IS', 'PT', 'IM',
       'AN', 'SLOVENIA', 'CT', 'SV', 'Roma', 'TN', 'VI', 'PR', 'Ancona',
       'BS', 'RM', 'CN', 'VR', 'BL', 'Trieste, Gorizia, Udine', 'PN',
       'AV', 'Livorno', 'AL', 'LC', 'Reggio di Calabria, Messina', 'TE',
       'TS', 'AUSTRIA, GERMANIA', 'CH', 'Savona', 'CROAZIA', 'FC', 'PI',
       'VT', 'GR', 'BO', 'AR', 'AUSTRIA', 'AP', 'RN', 'Chieti', 'GE',
       'FM', 'Pesaro-Urbino', 'ME', 'CZ', 'Catanzaro', 'Lecce',
       'Macerata', 'Imperia', 'MO', 'Fermo', 'LT', 'CO',
       'Rimini, Forli-Cesena', 'PV', 'MONTENEGRO', 'Salerno', 'BZ', 'RC',
       'CE', 'Catanzaro, Vibo Valentia, Reggio di Calabria', 'Teramo',
       'PC', 'Barletta', 'VV', 'Messina', 'SVIZZERA', 'Genova', 'CB',
       'SI', 'RE', 'LU', 'MN', 'P

In [ ]:
# Carica un file CSV contenente l'associazione tra province e regioni italiane
# Il file dovrebbe avere almeno due colonne, ad esempio: 'province' e 'regione'
province_regioni = pd.read_csv("../data/province_regioni_italia.csv")

In [13]:
province_regioni

,Provincia,Regione,Sigla Provincia
0,Agrigento,Sicilia,AG
1,Alessandria,Piemonte,AL
2,Ancona,Marche,AN
3,Aosta,Valle d'Aosta,AO
4,Arezzo,Toscana,AR
...,...,...,...
105,Vercelli,Piemonte,VC
106,Verona,Veneto,VR
107,Vibo Valentia,Calabria,VV
108,Vicenza,Veneto,VI


In [14]:
province_regioni['Sigla Provincia'] = province_regioni['Sigla Provincia'].fillna('NA')

In [ ]:
# Crea un dizionario di mapping da nome completo della provincia a sigla (es. "Napoli" → "NA")
mappa_province = dict(zip(province_regioni['Provincia'], province_regioni['Sigla Provincia']))

# Sostituisce i nomi delle province nella colonna 'province' con le rispettive sigle, se presenti nel mapping
# Se una provincia non è nel dizionario, mantiene il valore originale (grazie a fillna)
data['province'] = data['province'].map(mappa_province).fillna(data['province'])

In [16]:
data['province'].unique()

array(['AQ', 'FR', 'MC', 'TO', 'PG', 'KR', 'FRANCIA', 'CT', 'UD', 'FG',
       'VE', 'RI', 'CS', 'EN', nan, 'FI', 'PZ', 'BG', 'VB', 'SA',
       'ALBANIA', 'LE', 'TR', 'MT', 'SLOVENIA, CROAZIA', 'MS', 'BN',
       'MARE', 'PU', 'IS', 'PT', 'IM', 'AN', 'SLOVENIA', 'SV', 'RM', 'TN',
       'VI', 'PR', 'BS', 'CN', 'VR', 'BL', 'Trieste, Gorizia, Udine',
       'PN', 'AV', 'LI', 'AL', 'LC', 'Reggio di Calabria, Messina', 'TE',
       'TS', 'AUSTRIA, GERMANIA', 'CH', 'CROAZIA', 'FC', 'PI', 'VT', 'GR',
       'BO', 'AR', 'AUSTRIA', 'AP', 'RN', 'GE', 'FM', 'ME', 'CZ', 'MO',
       'LT', 'CO', 'Rimini, Forli-Cesena', 'PV', 'MONTENEGRO', 'BZ', 'RC',
       'CE', 'Catanzaro, Vibo Valentia, Reggio di Calabria', 'PC', 'BT',
       'VV', 'SVIZZERA', 'CB', 'SI', 'RE', 'LU', 'MN', 'PO', 'PA', 'FE',
       'TV', 'TP', 'AUSTRIA, SVIZZERA', 'GO', 'Salerno, Potenza',
       'AUSTRIA, SLOVENIA', 'NA', 'SP', 'VC', 'CR',
       'ITALIA, AUSTRIA, SLOVENIA', 'BA', 'AO', 'PD',
       'Massa-Carrara, Lucca, Pisa

In [ ]:
# Imposta a NaN tutte le province che non sono presenti nell'elenco ufficiale delle sigle di provincia
# (~data['province'].isin(...)) seleziona le righe in cui la sigla di provincia NON è tra quelle valide
data.loc[~data['province'].isin(province_regioni['Sigla Provincia']), 'province'] = np.nan

In [18]:
data['province'].unique()

array(['AQ', 'FR', 'MC', 'TO', 'PG', 'KR', nan, 'CT', 'UD', 'FG', 'VE',
       'RI', 'CS', 'EN', 'FI', 'PZ', 'BG', 'VB', 'SA', 'LE', 'TR', 'MT',
       'MS', 'BN', 'PU', 'IS', 'PT', 'IM', 'AN', 'SV', 'RM', 'TN', 'VI',
       'PR', 'BS', 'CN', 'VR', 'BL', 'PN', 'AV', 'LI', 'AL', 'LC', 'TE',
       'TS', 'CH', 'FC', 'PI', 'VT', 'GR', 'BO', 'AR', 'AP', 'RN', 'GE',
       'FM', 'ME', 'CZ', 'MO', 'LT', 'CO', 'PV', 'BZ', 'RC', 'CE', 'PC',
       'BT', 'VV', 'CB', 'SI', 'RE', 'LU', 'MN', 'PO', 'PA', 'FE', 'TV',
       'TP', 'GO', 'NA', 'SP', 'VC', 'CR', 'BA', 'AO', 'PD', 'BI', 'LO',
       'MI', 'PE', 'RA', 'AG', 'AT', 'RO', 'SR', 'SO', 'NO', 'BR', 'CL',
       'VA', 'MB', 'TA', 'OT', 'NU', 'CA', 'VS', 'OG', 'OR', 'SS'],
      dtype=object)

In [ ]:
# Esegue un merge tra il DataFrame 'data' e il DataFrame 'province_regioni'
# utilizzando la colonna 'province' di 'data' e 'Sigla Provincia' di 'province_regioni' come chiavi
# Il join è di tipo 'left', quindi mantiene tutte le righe di 'data'
data = data.merge(province_regioni, left_on='province', right_on='Sigla Provincia', how='left')

# Rimuove la colonna 'Sigla Provincia' duplicata, ormai ridondante dopo il merge
data = data.drop(columns=['Sigla Provincia'])

# Rinomina le colonne per chiarezza e coerenza:
# - 'province' diventa 'province_code' (la sigla, es. 'NA')
# - 'Provincia' diventa 'province_name' (nome completo, es. 'Napoli')
# - 'Regione' diventa 'region' (nome della regione, es. 'Campania')
data = data.rename(columns={
    'province': 'province_code',
    'Provincia': 'province_name',
    'Regione': 'region'
})

In [20]:
superfici_regioni = pd.read_csv("../data/superfici_regioni_italia.csv")

In [21]:
superfici_regioni

,Regione,Superficie (km²)
0,Abruzzo,10831
1,Basilicata,10073
2,Calabria,15221
3,Campania,13671
4,Emilia-Romagna,22453
5,Friuli-Venezia Giulia,7924
6,Lazio,17232
7,Liguria,5416
8,Lombardia,23863
9,Marche,9427


In [ ]:
# Esegue un merge tra il DataFrame 'data' e 'superfici_regioni'
# Unisce i dati sulla base del nome della regione: 'region' in 'data' e 'Regione' in 'superfici_regioni'
# Il join è di tipo 'left', quindi mantiene tutte le righe di 'data'
data = data.merge(
    superfici_regioni,
    how='left',
    left_on='region',
    right_on='Regione'
)

# Rinomina la colonna contenente la superficie regionale in km² in 'region_surface'
data = data.rename(columns={'Superficie (km²)': 'region_surface'})

# Elimina la colonna 'Regione' proveniente da 'superfici_regioni', ormai ridondante
data = data.drop(columns=['Regione'])

In [ ]:
# Conta il numero di terremoti per ciascuna regione
# Il risultato è una colonna con valori ripetuti per ogni riga appartenente alla stessa regione
data['region_earthquakes'] = data.groupby('region')['region'].transform('count')

# Calcola la densità di terremoti per regione (numero di eventi diviso la superficie in km²)
data['region_density'] = data['region_earthquakes'] / data['region_surface']

# Rimuove le colonne intermedie non più necessarie
data = data.drop(columns=['region_earthquakes', 'region_surface'])

In [ ]:
# Definisce le regioni italiane suddivise per macroarea geografica
nord = [
    'Emilia-Romagna', 'Friuli-Venezia Giulia', 'Liguria', 'Lombardia',
    'Piemonte', 'Trentino-Alto Adige', "Valle d'Aosta", 'Veneto'
]

centro = ['Lazio', 'Marche', 'Toscana', 'Umbria']

sud = [
    'Abruzzo', 'Basilicata', 'Calabria', 'Campania', 'Molise',
    'Puglia', 'Sardegna', 'Sicilia'
]

# Assegna a ciascuna riga una macroarea geografica (Nord, Centro, Sud) in base alla regione
# Se la regione non è riconosciuta, assegna None
data['area'] = data['region'].apply(lambda r: (
    'Nord' if r in nord else
    'Centro' if r in centro else
    'Sud' if r in sud else
    None
))

In [25]:
data

,time,magnitude,place,depth,latitude,longitude,year,month,day,hour,period,season,decade,province_code,province_name,region,region_density,area
0,1985-01-02 22:57:43.090,2.6,6 km S Civitella Alfedena (AQ),5.8,41.7150,13.9410,1985,1,2,22,night,winter,1985–1994,AQ,L'Aquila,Abruzzo,4.125473,Sud
1,1985-01-03 00:46:16.990,2.5,6 km NE Picinisco (FR),6.1,41.6770,13.9310,1985,1,3,0,night,winter,1985–1994,FR,Frosinone,Lazio,2.473480,Centro
2,1985-01-05 08:32:41.900,2.6,3 km NW Fiastra (MC),2.8,43.0560,13.1390,1985,1,5,8,day,winter,1985–1994,MC,Macerata,Marche,10.147873,Centro
3,1985-01-05 10:13:26.800,2.2,1 km E San Raffaele Cimena (TO),20.8,45.1500,7.8600,1985,1,5,10,day,winter,1985–1994,TO,Torino,Piemonte,0.236617,Nord
4,1985-01-06 03:52:23.380,2.4,3 km W Gubbio (PG),10.0,43.3570,12.5370,1985,1,6,3,night,winter,1985–1994,PG,Perugia,Umbria,12.044660,Centro
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
441817,2024-12-31 22:00:34.100,1.2,5 km NE Barberino di Mugello (FI),10.2,44.0287,11.2822,2024,12,31,22,night,winter,2015–2024,FI,Firenze,Toscana,0.869491,Centro
441818,2024-12-31 22:51:12.920,1.1,4 km NE Barberino di Mugello (FI),10.3,44.0262,11.2815,2024,12,31,22,night,winter,2015–2024,FI,Firenze,Toscana,0.869491,Centro
441819,2024-12-31 23:29:28.530,1.0,5 km NE Barberino di Mugello (FI),9.1,44.0267,11.2852,2024,12,31,23,night,winter,2015–2024,FI,Firenze,Toscana,0.869491,Centro
441820,2024-12-31 23:39:56.600,0.7,4 km NE Barberino di Mugello (FI),10.1,44.0273,11.2777,2024,12,31,23,night,winter,2015–2024,FI,Firenze,Toscana,0.869491,Centro


### Aggiunta di features legate a magnitudo e profondità

In [ ]:
# Definisce le condizioni per classificare i terremoti in base alla magnitudo, secondo le categorie INGV:
# < 2.0: bassa intensità
# 2.0–2.9: intensità medio-bassa
# 3.0–3.9: media
# 4.0–5.4: medio-alta
# ≥ 5.5: alta intensità
conditions = [
    data['magnitude'] < 2.0,
    (data['magnitude'] >= 2.0) & (data['magnitude'] < 3.0),
    (data['magnitude'] >= 3.0) & (data['magnitude'] < 4.0),
    (data['magnitude'] >= 4.0) & (data['magnitude'] < 5.5),
    data['magnitude'] >= 5.5
]

# Etichette corrispondenti ai livelli di potenza sismica
labels = ['low', 'mid-low', 'mid', 'mid-high', 'high']

# Applica le etichette alla colonna "power" in base alla condizione soddisfatta
# Se nessuna condizione è soddisfatta (caso raro), assegna 'unknown'
data["power"] = np.select(conditions, labels, default='unknown')

In [ ]:
# Calcola lo Z-score (standardizzazione) della magnitudo:
# misura quanto ciascun valore si discosta dalla media, in unità di deviazione standard
data['mag_z'] = (data['magnitude'] - data['magnitude'].mean()) / data['magnitude'].std()

# Calcola lo Z-score anche per la profondità dei terremoti
data['depth_z'] = (data['depth'] - data['depth'].mean()) / data['depth'].std()

In [28]:
data

,time,magnitude,place,depth,latitude,longitude,year,month,day,hour,...,season,decade,province_code,province_name,region,region_density,area,power,mag_z,depth_z
0,1985-01-02 22:57:43.090,2.6,6 km S Civitella Alfedena (AQ),5.8,41.7150,13.9410,1985,1,2,22,...,winter,1985–1994,AQ,L'Aquila,Abruzzo,4.125473,Sud,mid-low,1.659074,-0.350789
1,1985-01-03 00:46:16.990,2.5,6 km NE Picinisco (FR),6.1,41.6770,13.9310,1985,1,3,0,...,winter,1985–1994,FR,Frosinone,Lazio,2.473480,Centro,mid-low,1.506194,-0.334980
2,1985-01-05 08:32:41.900,2.6,3 km NW Fiastra (MC),2.8,43.0560,13.1390,1985,1,5,8,...,winter,1985–1994,MC,Macerata,Marche,10.147873,Centro,mid-low,1.659074,-0.508877
3,1985-01-05 10:13:26.800,2.2,1 km E San Raffaele Cimena (TO),20.8,45.1500,7.8600,1985,1,5,10,...,winter,1985–1994,TO,Torino,Piemonte,0.236617,Nord,mid-low,1.047554,0.439648
4,1985-01-06 03:52:23.380,2.4,3 km W Gubbio (PG),10.0,43.3570,12.5370,1985,1,6,3,...,winter,1985–1994,PG,Perugia,Umbria,12.044660,Centro,mid-low,1.353314,-0.129467
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
441817,2024-12-31 22:00:34.100,1.2,5 km NE Barberino di Mugello (FI),10.2,44.0287,11.2822,2024,12,31,22,...,winter,2015–2024,FI,Firenze,Toscana,0.869491,Centro,low,-0.481247,-0.118928
441818,2024-12-31 22:51:12.920,1.1,4 km NE Barberino di Mugello (FI),10.3,44.0262,11.2815,2024,12,31,22,...,winter,2015–2024,FI,Firenze,Toscana,0.869491,Centro,low,-0.634127,-0.113658
441819,2024-12-31 23:29:28.530,1.0,5 km NE Barberino di Mugello (FI),9.1,44.0267,11.2852,2024,12,31,23,...,winter,2015–2024,FI,Firenze,Toscana,0.869491,Centro,low,-0.787007,-0.176893
441820,2024-12-31 23:39:56.600,0.7,4 km NE Barberino di Mugello (FI),10.1,44.0273,11.2777,2024,12,31,23,...,winter,2015–2024,FI,Firenze,Toscana,0.869491,Centro,low,-1.245647,-0.124197


In [29]:
data.to_csv('../data/terremoti_ingv_1985_2025_extended.csv', index=False)